In [ ]:
{
  "nbformat": 4,
  "nbformat_minor": 0,
  "metadata": {
    "colab": {
      "provenance": []
    },
    "kernelspec": {
      "name": "python3",
      "display_name": "Python 3"
    },
    "language_info": {
      "name": "python"
    }
  },
  "cells": [
    {
      "cell_type": "code",
      "source": [
        "pip install ftfy"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "Av2ppt85K9YI",
        "outputId": "8a5ed52c-f795-49d6-93d1-bae619245e71"
      },
      "execution_count": 1,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Collecting ftfy\n",
            "  Downloading ftfy-6.1.3-py3-none-any.whl (53 kB)\n",
            "\u001b[?25l     \u001b[90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[32m0.0/53.4 kB\u001b[0m \u001b[31m?\u001b[0m eta \u001b[36m-:--:--\u001b[0m\r\u001b[2K     \u001b[91m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[90m╺\u001b[0m\u001b[90m━\u001b[0m \u001b[32m51.2/53.4 kB\u001b[0m \u001b[31m1.5 MB/s\u001b[0m eta \u001b[36m0:00:01\u001b[0m\r\u001b[2K     \u001b[90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[32m53.4/53.4 kB\u001b[0m \u001b[31m859.8 kB/s\u001b[0m eta \u001b[36m0:00:00\u001b[0m\n",
            "\u001b[?25hRequirement already satisfied: wcwidth<0.3.0,>=0.2.12 in /usr/local/lib/python3.10/dist-packages (from ftfy) (0.2.12)\n",
            "Installing collected packages: ftfy\n",
            "Successfully installed ftfy-6.1.3\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "import torch.nn as nn\n",
        "from transformers import GPT2LMHeadModel\n",
        "import torch\n",
        "import torch.nn as nn\n",
        "\n",
        "class ClipCaptionModel(nn.Module):\n",
        "\n",
        "    def forward(self, tokens, prefix, mask= None, labels= None):\n",
        "        embedding_text = self.gpt.transformer.wte(tokens)\n",
        "        prefix_projections = self.clip_project(prefix).view(-1, self.prefix_length, self.gpt_embedding_size)\n",
        "        embedding_cat = torch.cat((prefix_projections, embedding_text), dim=1)\n",
        "        if labels is not None:\n",
        "            dummy_token = self.get_dummy_token(tokens.shape[0], tokens.device)\n",
        "            labels = torch.cat((dummy_token, tokens), dim=1)\n",
        "        out = self.gpt(inputs_embeds=embedding_cat, labels=labels, attention_mask=mask)\n",
        "        return out\n",
        "\n",
        "    def __init__(self, prefix_length, prefix_size= 512):\n",
        "        super(ClipCaptionModel, self).__init__()\n",
        "        self.prefix_length = prefix_length\n",
        "        self.gpt = GPT2LMHeadModel.from_pretrained('gpt2')\n",
        "        self.gpt_embedding_size = self.gpt.transformer.wte.weight.shape[1]\n",
        "        if prefix_length > 10:  # not enough memory\n",
        "            self.clip_project = nn.Linear(prefix_size, self.gpt_embedding_size * prefix_length)\n",
        "        else:\n",
        "            self.clip_project = MLP(\n",
        "                (\n",
        "                    prefix_size,\n",
        "                    (self.gpt_embedding_size * prefix_length) // 2,\n",
        "                    self.gpt_embedding_size * prefix_length,\n",
        "                )\n",
        "            )\n",
        "\n"
      ],
      "metadata": {
        "id": "ivYstr6MIidz"
      },
      "execution_count": 17,
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "import torch.nn as nn\n",
        "import torch.nn.functional as F\n",
        "import torch\n",
        "class MLP(nn.Module):\n",
        "    def forward(self, x):\n",
        "        return self.model(x)\n",
        "\n",
        "    def __init__(self, sizes, bias=True, act=nn.Tanh):\n",
        "        super(MLP, self).__init__()\n",
        "        layers = []\n",
        "        for i in range(len(sizes) -1):\n",
        "            layers.append(nn.Linear(sizes[i], sizes[i + 1], bias=bias))\n",
        "            if i < len(sizes) - 2:\n",
        "                layers.append(act())\n",
        "        self.model = nn.Sequential(*layers)\n"
      ],
      "metadata": {
        "id": "81GJ7LX6InBy"
      },
      "execution_count": 18,
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "# def generate_simple():\n",
        "#   model.eval()\n",
        "#   stop_token_index = tokenizer.encode(stop_token)[0]\n",
        "#   tokens = None\n",
        "#   scores = None\n",
        "#   seq_lens = torch.ones(beam_size, device=device)\n",
        "#   is_stopped = torch.zeros(beam_size, device=device,dtype=torch.bool)\n",
        "#   with torch.no_grad():\n",
        "#     if embed is not None:\n",
        "#       generated = embed\n",
        "#     else:\n",
        "#       if tokens is None:\n",
        "#         tokens = torch.tensor(tokenizer.encode(prompt))\n",
        "#         tokens = tokens.unsqueeze(0)\n",
        "#         generated = model.gpt.transformer.wte(tokens)\n",
        "#   for i in range(max_len):\n",
        "#     outputs = model.gpt(inputs_embeds = generated)\n",
        "import torch\n",
        "import numpy as np\n",
        "\n",
        "def generate_beam(model, tokenizer, beam_size: int = 5, prompt=None, embed=None,\n",
        "                  entry_length=67, temperature=1., stop_token: str = '.'):\n",
        "\n",
        "    model.eval()\n",
        "    stop_token_index = tokenizer.encode(stop_token)[0]\n",
        "    tokens = None\n",
        "    scores = None\n",
        "    device = next(model.parameters()).device\n",
        "    seq_lengths = torch.ones(beam_size, device=device)\n",
        "    is_stopped = torch.zeros(beam_size, device=device, dtype=torch.bool)\n",
        "    with torch.no_grad():\n",
        "        if embed is not None:\n",
        "            generated = embed\n",
        "        else:\n",
        "            if tokens is None:\n",
        "                tokens = torch.tensor(tokenizer.encode(prompt))\n",
        "                tokens = tokens.unsqueeze(0).to(device)\n",
        "                generated = model.gpt.transformer.wte(tokens)\n",
        "        for i in range(entry_length):\n",
        "            outputs = model.gpt(inputs_embeds=generated)\n",
        "            logits = outputs.logits\n",
        "            logits = logits[:, -1, :] / (temperature if temperature > 0 else 1.0)\n",
        "            logits = logits.softmax(-1).log()\n",
        "            if scores is None:\n",
        "                scores, next_tokens = logits.topk(beam_size, -1)\n",
        "                generated = generated.expand(beam_size, *generated.shape[1:])\n",
        "                next_tokens, scores = next_tokens.permute(1, 0), scores.squeeze(0)\n",
        "                if tokens is None:\n",
        "                    tokens = next_tokens\n",
        "                else:\n",
        "                    tokens = tokens.expand(beam_size, *tokens.shape[1:])\n",
        "                    tokens = torch.cat((tokens, next_tokens), dim=1)\n",
        "            else:\n",
        "                logits[is_stopped] = -float(np.inf)\n",
        "                logits[is_stopped, 0] = 0\n",
        "                scores_sum = scores[:, None] + logits\n",
        "                seq_lengths[~is_stopped] += 1\n",
        "                scores_sum_average = scores_sum / seq_lengths[:, None]\n",
        "                scores_sum_average, next_tokens = scores_sum_average.view(-1).topk(beam_size, -1)\n",
        "                next_tokens_source = next_tokens // scores_sum.shape[1]\n",
        "                seq_lengths = seq_lengths[next_tokens_source]\n",
        "                next_tokens = next_tokens % scores_sum.shape[1]\n",
        "                next_tokens = next_tokens.unsqueeze(1)\n",
        "                tokens = tokens[next_tokens_source]\n",
        "                tokens = torch.cat((tokens, next_tokens), dim=1)\n",
        "                generated = generated[next_tokens_source]\n",
        "                scores = scores_sum_average * seq_lengths\n",
        "                is_stopped = is_stopped[next_tokens_source]\n",
        "            next_token_embed = model.gpt.transformer.wte(next_tokens.squeeze()).view(generated.shape[0], 1, -1)\n",
        "            generated = torch.cat((generated, next_token_embed), dim=1)\n",
        "            is_stopped = is_stopped + next_tokens.eq(stop_token_index).squeeze()\n",
        "            if is_stopped.all():\n",
        "                break\n",
        "    scores = scores / seq_lengths\n",
        "    output_list = tokens.cpu().numpy()\n",
        "    output_texts = [tokenizer.decode(output[:int(length)]) for output, length in zip(output_list, seq_lengths)]\n",
        "    order = scores.argsort(descending=True)\n",
        "    output_texts = [output_texts[i] for i in order]\n",
        "    return output_texts\n"
      ],
      "metadata": {
        "id": "eBrz0W5NIrf-"
      },
      "execution_count": 19,
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "import gdown\n",
        "\n",
        "class Downloader(object):\n",
        "    def __init__(self):\n",
        "        pass\n",
        "    def download_file(self, file_id, file_dst):\n",
        "        url = f\"https://drive.google.com/uc?id={file_id}\"\n",
        "        gdown.download(url, output=file_dst)\n"
      ],
      "metadata": {
        "id": "IpC3PxpkIyt1"
      },
      "execution_count": 20,
      "outputs": []
    },
    {
      "cell_type": "code",
      "execution_count": 23,
      "metadata": {
        "id": "7yMgGaJUOul6"
      },
      "outputs": [],
      "source": [
        "import gdown\n",
        "import subprocess\n",
        "subprocess.run([\"git\", \"clone\", \"https://github.com/openai/CLIP.git\"])\n",
        "subprocess.run([\"pip\", \"install\", \"CLIP\"])\n",
        "\n",
        "from CLIP.clip import *\n",
        "import os\n",
        "from torch import nn\n",
        "import numpy as np\n",
        "import torch\n",
        "import torch.nn.functional as nnf\n",
        "import sys\n",
        "from transformers import GPT2Tokenizer, GPT2LMHeadModel\n",
        "from tqdm import tqdm, trange\n",
        "import skimage.io as io\n",
        "import PIL.Image\n",
        "from flask import Flask, redirect, url_for, request, render_template\n",
        "from werkzeug.utils import secure_filename\n",
        "downloader = Downloader()\n",
        "\n",
        "device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n",
        "\n",
        "current_directory = os.getcwd()\n",
        "save_path = os.path.join(os.path.dirname(current_directory), \"Caption-Craft/pretrained_models\")\n",
        "os.makedirs(save_path, exist_ok=True)\n",
        "model_path = os.path.join(save_path, 'model_weights.pkl')\n",
        "\n",
        "if not os.path.exists(model_path):\n",
        "    downloader.download_file(\"14pXWwB4Zm82rsDdvbGguLfx9F8aM7ovT\", model_path)\n",
        "clip_model, preprocess = load(\"ViT-B/32\", device=device, jit=False)\n",
        "tokenizer = GPT2Tokenizer.from_pretrained(\"gpt2\")\n",
        "\n",
        "prefix_length = 10\n",
        "\n",
        "model = ClipCaptionModel(prefix_length)\n",
        "\n",
        "model.load_state_dict(torch.load(model_path, map_location=\"cpu\"),strict=False)\n",
        "\n",
        "model = model.eval()\n",
        "model = model.to(device)\n",
        "\n",
        "def model_predict(img, model, path=False):\n",
        "    if path:\n",
        "        im = io.imread(img)\n",
        "        img = PIL.Image.fromarray(im)\n",
        "    pil_image = img\n",
        "    image = preprocess(pil_image).unsqueeze(0).to(device)\n",
        "    with torch.no_grad():\n",
        "        prefix = clip_model.encode_image(image).to(device, dtype=torch.float32)\n",
        "        prefix_embed = model.clip_project(prefix).reshape(1, prefix_length, -1)\n",
        "        generated_text_prefix = generate_beam(model, tokenizer, embed=prefix_embed)[0]\n",
        "    return generated_text_prefix\n",
        "\n",
        "\n"
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "model_predict(\"/content/pm.jpeg\",model,path=True)"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/",
          "height": 36
        },
        "id": "kJtRFOjNI2_u",
        "outputId": "d4254c66-d9a5-4d6b-c5f1-41028db2e14f"
      },
      "execution_count": 28,
      "outputs": [
        {
          "output_type": "execute_result",
          "data": {
            "text/plain": [
              "'soccer player vies for the ball with football player during their football match at the stadium.'"
            ],
            "application/vnd.google.colaboratory.intrinsic+json": {
              "type": "string"
            }
          },
          "metadata": {},
          "execution_count": 28
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [],
      "metadata": {
        "id": "pG3jv1n2OfGq"
      },
      "execution_count": 16,
      "outputs": []
    }
  ]
}